In [ ]:
# eval.ipynb
# Deferred evaluation: run AFTER run.ipynb finishes training, on the SAME pod.
# Training only emitted slim probe checkpoints to a queue (no probe ran during
# training, to avoid GPU contention). This notebook drains that queue (Prob),
# runs the final pooling-vs-VICReg eval, then archives everything.
REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
LOG = "/workspace/stable_query_latent_logs/pipeline.log"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
QUEUE_DIR = OUT_DIR + "/probe_queue"
print('repo:', REPO)
print('out :', OUT_DIR)
print('log :', LOG)


In [ ]:
# Pull latest code (probe_worker / eval logic) before evaluating.
import os

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git pull origin main
!git rev-parse HEAD


In [ ]:
# Start GPU + CPU + RAM + disk I/O monitor (same as run.ipynb).
import subprocess, threading, time, psutil
from pathlib import Path

stop = False

def _read_int(path):
    try:
        text = Path(path).read_text().strip()
        if text == "max":
            return None
        return int(text)
    except Exception:
        return None

def get_memory_status():
    limit = _read_int("/sys/fs/cgroup/memory.max")
    used = _read_int("/sys/fs/cgroup/memory.current")
    if limit is None or used is None:
        limit = _read_int("/sys/fs/cgroup/memory/memory.limit_in_bytes")
        used = _read_int("/sys/fs/cgroup/memory/memory.usage_in_bytes")
    if limit and used and limit < 10**18:
        return used / limit * 100, used / 1024**3, limit / 1024**3, "cgroup"
    vm = psutil.virtual_memory()
    return vm.percent, vm.used / 1024**3, vm.total / 1024**3, "host"

def get_gpu_status():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            timeout=3,
        ).stdout.strip()
        parts = [p.strip() for p in out.splitlines()[0].split(",")]
        return f"{float(parts[0]):.0f}%, {float(parts[1])/1024:.1f}/{float(parts[2])/1024:.1f} GiB"
    except Exception as e:
        return f"n/a ({e})"

def monitor(interval=5):
    last_disk = psutil.disk_io_counters()
    last_t = time.time()
    psutil.cpu_percent(interval=None)
    while not stop:
        gpu = get_gpu_status()
        cpu = psutil.cpu_percent(interval=None)
        ram_pct, ram_used, ram_total, ram_source = get_memory_status()
        now_disk = psutil.disk_io_counters()
        now_t = time.time()
        dt = max(now_t - last_t, 1e-6)
        read_mb = (now_disk.read_bytes - last_disk.read_bytes) / 1e6 / dt
        write_mb = (now_disk.write_bytes - last_disk.write_bytes) / 1e6 / dt
        last_disk, last_t = now_disk, now_t
        print(
            f"[gpu] {gpu} | [cpu] {cpu:.0f}% | "
            f"[ram:{ram_source}] {ram_pct:.0f}% ({ram_used:.1f}/{ram_total:.1f} GiB) | "
            f"[disk] R {read_mb:.1f} MB/s W {write_mb:.1f} MB/s",
            flush=True,
        )
        time.sleep(interval)

threading.Thread(target=monitor, daemon=True).start()


In [ ]:
# Clear GPU memory.
import gc, torch
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Prob: drain the probe queue in ONE pass (training is done, so the backlog is
# complete and static). This fills every <combo>/dual_probe_history.tsv with the
# convergence curve. GPU is fully free now, so there is no training contention.
!python -u VICReg_review/probe_worker.py \
  --queue-dir {QUEUE_DIR} \
  --device cuda \
  --run-once \
  --logout-address {LOG}


In [ ]:
# Clear GPU memory.
import gc, torch
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Final eval: pooling-vs-VICReg comparison on the best full-train checkpoint.
# --skip-train skips all training; --eval-mode final_best runs the comparison.
# --calib-mode off skips the (training-only) OOM calibration. Grid flags mirror
# run.ipynb so the sweep can locate the right best checkpoint.
!python -u VICReg_review/sweep_cloud.py \
  --h5 game_review_data/embedding_h5.h5 \
  --out-dir {OUT_DIR} \
  --train-game-counts 50 100 200 500 1000 1500 2000 \
  --sample-fractions 0.8 0.6 0.4 0.2 \
  --output-dims 18 36 64 72 \
  --latent-scales 1 2 4 \
  --base-num-latents 256 \
  --expander-dim 128 \
  --expander-hidden 128 \
  --epochs 30 \
  --batch-size 128 \
  --skip-train \
  --eval-mode final_best \
  --calib-mode off \
  --probe-queue-dir {QUEUE_DIR} \
  --train-game-anchor-appids "1091500,1385380" \
  --logout-address {LOG}


In [ ]:
# Collect outputs (archive heads, H5s, manifests + paper handoff).
stop = True

from datetime import datetime
from pathlib import Path
import json
import subprocess
import shutil

repo = Path(REPO)
out = Path('/workspace') / 'stable_query_latent_artifacts' / datetime.now().strftime('%Y%m%d_%H%M%S')
out.mkdir(parents=True, exist_ok=True)

def run_capture(cmd):
    return subprocess.run(cmd, cwd=repo, capture_output=True, text=True).stdout.strip()

def copy_item(rel: str):
    src = repo / rel
    dst = out / rel
    if not src.exists():
        print(f'skip missing: {src}')
        return None
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        kind = 'dir'
    else:
        shutil.copy2(src, dst)
        kind = 'file'
    print(f'copied: {rel}')
    return {'item': rel, 'kind': kind}

items = [
    'VICReg_review/heads',
    'game_review_data/embedding_h5.h5',
    'game_review_data/embedding_h5.h5.incloud_manifest.json',
    'game_review_data/build_new_gamedata/text_h5.h5',
    'game_review_data/build_new_gamedata/text_h5.h5.manifest.json',
]
manifest = [x for x in (copy_item(rel) for rel in items) if x is not None]

log_src = Path(LOG)
if log_src.exists():
    log_dst = out / 'pipeline.log'
    shutil.copy2(log_src, log_dst)
    manifest.append({'item': str(log_src), 'kind': 'file', 'copied_as': 'pipeline.log'})
    print(f'copied log: {log_src}')
else:
    print(f'skip missing log: {log_src}')

git_info = {
    'commit': run_capture(['git', 'rev-parse', 'HEAD']),
    'branch': run_capture(['git', 'branch', '--show-current']),
    'status_short': run_capture(['git', 'status', '--short']),
    'remote': run_capture(['git', 'remote', '-v']),
}
(out / 'git_info.json').write_text(json.dumps(git_info, ensure_ascii=False, indent=2), encoding='utf-8')

sweep_dir = out / 'VICReg_review' / 'heads' / 'cloud_full_sweep_a100'
paper_files = [
    'DATA_VIEW_SWEEP_REPORT.md',
    'data_view_sweep_summary.csv',
    'data_view_sweep_summary.json',
    'sweep_manifest.json',
    'calib.json',
    # final_best eval-mode writes the real pooling-vs-VICReg comparison here,
    # not into per-combo eval_report.json files, so point the handoff at it.
    'final_best_eval/final_best_eval.json',
    'final_best_eval/eval_report.json',
    'raw_test_data/training_manifests.csv',
    'raw_test_data/probe_summary.csv',
    'raw_test_data/recommendation_probe.csv',
    'raw_test_data/identity_summary.csv',
    'raw_test_data/identity_retrieval_details.csv',
    'raw_test_data/identity_pair_cosine_details.csv',
    'raw_test_data/tag_freq_floor_details.csv',
    'raw_test_data/tag_top_bottom_details.csv',
    'raw_test_data/tag_fold_details.csv',
]
available_paper_files = [rel for rel in paper_files if (sweep_dir / rel).exists()]
missing_paper_files = [rel for rel in paper_files if not (sweep_dir / rel).exists()]

# Per-combo in-training convergence histories (one row per probe epoch).
probe_history_files = sorted(
    str(p.relative_to(sweep_dir)).replace('\\', '/')
    for p in sweep_dir.glob('*/dual_probe_history.tsv')
)

handoff = [
    '# Paper Handoff',
    '',
    f'- Archive: `{out}`',
    f'- Git commit: `{git_info["commit"]}`',
    f'- Sweep dir: `{sweep_dir}`',
    f'- Pipeline log: `pipeline.log`' if (out / 'pipeline.log').exists() else '- Pipeline log: missing',
    '',
    '## Start Here',
    '',
    '1. Read `VICReg_review/heads/cloud_full_sweep_a100/DATA_VIEW_SWEEP_REPORT.md` for the high-level result.',
    '2. Read `final_best_eval/final_best_eval.json` for the pooling-vs-VICReg comparison on the best full-train model (the actual probe/identity numbers).',
    '3. Use `data_view_sweep_summary.csv` + `raw_test_data/training_manifests.csv` for the data-size x architecture grid (training loss/convergence per combo).',
    '4. Use `<combo>/dual_probe_history.tsv` for per-combo in-training convergence curves (sentiment/recommendation/tag probes every few epochs).',
    '5. Check `sweep_manifest.json` before interpreting incomplete runs; `calib.json` records the OOM-budget calibration used.',
    '',
    '## Available Paper Files',
    '',
]
handoff.extend(f'- `{rel}`' for rel in available_paper_files)
if missing_paper_files:
    handoff.extend(['', '## Missing Or Not Yet Generated', ''])
    handoff.extend(f'- `{rel}`' for rel in missing_paper_files)
handoff.extend([
    '',
    '## Convergence Probe Histories',
    '',
    f'- {len(probe_history_files)} per-combo files at `<combo>/dual_probe_history.tsv` '
    '(full list in `collection_manifest.json` -> `probe_history_files`).',
])
(out / 'paper_handoff.md').write_text('\n'.join(handoff) + '\n', encoding='utf-8')

(out / 'collection_manifest.json').write_text(
    json.dumps({'repo': str(repo), 'archive': str(out), 'git': git_info, 'items': manifest, 'paper_files': available_paper_files, 'missing_paper_files': missing_paper_files, 'probe_history_files': probe_history_files}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print(f'archive ready: {out}')
print(f'probe history files: {len(probe_history_files)}')
